# Architecture 6: BioClinicalBERT + LoRA + LightGBM Fusion (Full Triage Note)

**Key insight:** Previous architectures discarded PMH and medications entirely, and truncated HPI at 128 words.
The combined triage note (CC + vitals + full HPI + full PMH + full medications) is only ~270 tokens median —
fits within BioClinicalBERT's 512 token limit without any truncation.

**Changes from arch5_v3:**
1. **Full triage note** — CC + patient_info + initial_vitals + full HPI + full PMH + full past_medication.
2. **Source: PMH_v2** joined with features CSV for structured fields.
3. **MAX_LEN = 512** — covers 95%+ of triage notes.
4. **No leakage** — triage-time fields only (no diagnosis, labs, treatment).
5. **LoRA + LightGBM** — retained from arch5_v3.


## 1. Setup & Installs

In [ ]:
!pip install -q boto3 transformers peft scikit-learn "numpy==1.26.4"

## 2. Imports & Config

In [ ]:
import io
import json
import math
import random
import warnings

import boto3
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from lightgbm import LGBMClassifier, early_stopping
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── AWS / S3 ───────────────────────────────────────────────────────────────────
S3_BUCKET    = "ed-triage-capstone-group7"
FEAT_KEY     = "Data_Output/consolidated_dataset_features.csv"
PMH_KEY      = "Data_Output/consolidated_dataset_PMH_v2.csv"
SPLIT_PREFIX = "Data_Output/splits/arch6/"
MODEL_PREFIX = "models/arch6/"
BERT_MODEL   = "emilyalsentzer/Bio_ClinicalBERT"

# ── LoRA ───────────────────────────────────────────────────────────────────────
LORA_R              = 32   # increased from 16 — richer 512-token input needs more adapter capacity
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["query", "key", "value", "output.dense"]  # full attention block

# ── Training ───────────────────────────────────────────────────────────────────
NUM_CLASSES        = 3
MAX_LEN            = 512
BATCH_SIZE         = 16
ACCUMULATION_STEPS = 2
EPOCHS             = 25
PATIENCE           = 7
BERT_LR            = 5e-5
HEAD_LR            = 1.5e-4
HEAD_HIDDEN_DIM    = 96
HEAD_DROPOUT       = 0.4

# ── Device ────────────────────────────────────────────────────────────────────
AMP_DTYPE       = torch.float16
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AUTOCAST_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
scaler          = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 3. Load Data from S3

In [ ]:
# ── Debug mode — set True to run a cheap local sanity check ───────────────────
# Caps data to 300 rows, 2 epochs, skips S3 saves.
# Set False before running on SageMaker.
DEBUG_MODE = True

if DEBUG_MODE:
    EPOCHS             = 2
    PATIENCE           = 99   # don't early-stop in debug
    BATCH_SIZE         = 8
    ACCUMULATION_STEPS = 1
    print("DEBUG MODE ON — 300 rows, 2 epochs, no S3 saves")
else:
    print("PRODUCTION MODE — full data, S3 saves enabled")

In [ ]:
s3 = boto3.client("s3", region_name="us-east-1")

# Load features CSV — has structured vitals, pain_missing, engineered features
feat_obj = s3.get_object(Bucket=S3_BUCKET, Key=FEAT_KEY)
df       = pd.read_csv(io.BytesIO(feat_obj["Body"].read()))
df["stay_id"] = df["stay_id"].astype(int)

# Load PMH_v2 — richer text fields (full HPI, PMH, medications)
pmh_obj = s3.get_object(Bucket=S3_BUCKET, Key=PMH_KEY)
df_pmh  = pd.read_csv(io.BytesIO(pmh_obj["Body"].read()),
                      usecols=["stay_id", "HPI", "past_medical_history",
                                "past_medication", "initial_vitals"])
df_pmh["stay_id"] = df_pmh["stay_id"].astype(int)

# Inner join — replace text fields with richer PMH_v2 versions
df = df.merge(df_pmh, on="stay_id", how="inner", suffixes=("_feat", ""))

# Drop duplicate feat columns if any
for col in ["HPI", "past_medical_history", "initial_vitals"]:
    feat_col = f"{col}_feat"
    if feat_col in df.columns:
        df.drop(columns=[feat_col], inplace=True)

print(f"Merged rows: {len(df)}")
print(f"pain_missing present: {'pain_missing' in df.columns}")
print(f"past_medication non-null: {df['past_medication'].notna().sum()} / {len(df)}")
print(f"past_medical_history non-null: {df['past_medical_history'].notna().sum()} / {len(df)}")

# 4-class -> 3-class
target_map = {1: 0, 2: 1, 3: 2, 4: 2}
df["triage_3class"] = df["triage"].map(target_map).astype(int)

if DEBUG_MODE:
    df = df.groupby("triage", group_keys=False).apply(
        lambda x: x.sample(min(len(x), max(1, int(300 * len(x) / len(df)))), random_state=SEED)
    ).reset_index(drop=True)
    print(f"DEBUG: sampled {len(df)} rows")

print(f"\nTriage distribution:")
print(df["triage_3class"].value_counts().sort_index().rename(
    {0: "L1-Critical", 1: "L2-Emergent", 2: "L3-Urgent/LessUrgent"}
).to_string())


## 4. Full Triage Note Construction

In [ ]:
CLASS_NAMES = ["L1-Critical", "L2-Emergent", "L3-Urgent/LessUrgent"]

def clean(val):
    if pd.isna(val) or str(val).strip().lower() in ("nan", "none", ""):
        return ""
    return str(val).replace("\n", " ").strip()

def build_triage_note(row):
    """Full triage-time note. No word limits — tokenizer truncates at MAX_LEN=512."""
    parts = []
    if clean(row["patient_info"]):
        parts.append(f"Patient: {clean(row['patient_info'])}.")
    if clean(row["chiefcomplaint"]):
        cc = clean(row["chiefcomplaint"])
        parts.append(f"Chief complaint: {cc}.")
        parts.append(f"Presenting with {cc}.")
    if clean(row["initial_vitals"]):
        parts.append(f"Vitals: {clean(row['initial_vitals'])}.")
    if not row["pain_missing"] and pd.notna(row["pain"]):
        parts.append(f"Pain: {row['pain']:.0f}/10.")
    if clean(row["arrival_transport"]):
        parts.append(f"Arrival: {clean(row['arrival_transport'])}.")
    if clean(row["HPI"]):
        parts.append(f"History: {clean(row['HPI'])}.")
    if clean(row["past_medical_history"]):
        parts.append(f"Past medical history: {clean(row['past_medical_history'])}.")
    if clean(row["past_medication"]):
        parts.append(f"Medications: {clean(row['past_medication'])}.")
    return " ".join(parts)

df["triage_text"] = df.apply(build_triage_note, axis=1)

wc = df["triage_text"].str.split().str.len()
print("triage_text word counts:")
print(f"  median: {wc.median():.0f}")
print(f"  90th:   {wc.quantile(0.90):.0f}")
print(f"  95th:   {wc.quantile(0.95):.0f}")
print()
print("Sample (first row):")
print(df["triage_text"].iloc[0][:600] + "...")


In [ ]:
BASE_COLUMNS = [
    "stay_id", "triage", "triage_3class", "triage_text",
    "chiefcomplaint", "HPI", "past_medical_history", "past_medication",
    "initial_vitals", "arrival_transport",
    "pain", "pain_missing", "age",
    "temp_f", "heart_rate", "resp_rate", "spo2", "sbp", "dbp",
    "patient_info",
]

X = df[BASE_COLUMNS].copy()
y = df["triage_3class"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

total = len(df)
print(f"Train: {len(X_train)} ({len(X_train)/total:.1%})")
print(f"Val:   {len(X_val)}   ({len(X_val)/total:.1%})")
print(f"Test:  {len(X_test)}  ({len(X_test)/total:.1%})")
print()
print("Train class counts:")
print(pd.Series(y_train).value_counts().sort_index().rename(
    {0: CLASS_NAMES[0], 1: CLASS_NAMES[1], 2: CLASS_NAMES[2]}
).to_string())

if not DEBUG_MODE:
    def save_split_to_s3(split_df, y_split, key):
        out = split_df.copy()
        out["triage_3class"] = y_split
        buf = io.BytesIO()
        out.to_csv(buf, index=False)
        buf.seek(0)
        s3.put_object(Bucket=S3_BUCKET, Key=key, Body=buf.getvalue(), ContentType="text/csv")
        print(f"  Saved -> s3://{S3_BUCKET}/{key}  ({len(out)} rows)")
    print("\nSaving splits:")
    save_split_to_s3(X_train, y_train, f"{SPLIT_PREFIX}train.csv")
    save_split_to_s3(X_val,   y_val,   f"{SPLIT_PREFIX}val.csv")
    save_split_to_s3(X_test,  y_test,  f"{SPLIT_PREFIX}test.csv")
else:
    print("DEBUG: skipping S3 split saves")


## 5. Structured Feature Pipeline (LightGBM)

In [ ]:
RAW_VITALS = ["temp_f", "heart_rate", "resp_rate", "spo2", "sbp", "dbp"]
CLIP_BOUNDS = {
    "temp_f":     (85.0, 115.0),
    "heart_rate": (20.0, 250.0),
    "resp_rate":  (4.0,  60.0),
    "spo2":       (50.0, 100.0),
    "sbp":        (40.0, 300.0),
    "dbp":        (10.0, 200.0),
}
TRANSPORT_MAP = {"WALK IN": 0, "UNKNOWN": 1, "AMBULANCE": 2, "HELICOPTER": 3}

STRUCTURED_FEATURES = [
    "heart_rate", "sbp", "dbp", "resp_rate", "spo2", "temp_f",
    "shock_index", "map", "pulse_pressure",
    "news2_score", "mews_score",
    "age",
    "transport_ordinal",
    "pain", "pain_missing",
]

def compute_news2(row):
    score = 0
    rr = row["resp_rate"]
    if   rr <= 8:   score += 3
    elif rr <= 11:  score += 1
    elif rr <= 20:  score += 0
    elif rr <= 24:  score += 2
    else:           score += 3
    spo2 = row["spo2"]
    if   spo2 <= 91:  score += 3
    elif spo2 <= 93:  score += 2
    elif spo2 <= 95:  score += 1
    sbp = row["sbp"]
    if   sbp <= 90:   score += 3
    elif sbp <= 100:  score += 2
    elif sbp <= 110:  score += 1
    elif sbp <= 219:  score += 0
    else:             score += 3
    hr = row["heart_rate"]
    if   hr <= 40:   score += 3
    elif hr <= 50:   score += 1
    elif hr <= 90:   score += 0
    elif hr <= 110:  score += 1
    elif hr <= 130:  score += 2
    else:            score += 3
    temp = row["temp_f"]
    if   temp <= 95.0:    score += 3
    elif temp <= 96.8:    score += 1
    elif temp <= 100.4:   score += 0
    elif temp <= 102.2:   score += 1
    else:                 score += 2
    return score

def compute_mews(row):
    score = 0
    sbp = row["sbp"]
    if   sbp < 70:    score += 3
    elif sbp < 81:    score += 2
    elif sbp < 101:   score += 1
    elif sbp < 200:   score += 0
    else:             score += 2
    hr = row["heart_rate"]
    if   hr < 40:     score += 2
    elif hr < 51:     score += 1
    elif hr < 101:    score += 0
    elif hr < 111:    score += 1
    elif hr < 130:    score += 2
    else:             score += 3
    rr = row["resp_rate"]
    if   rr < 9:      score += 2
    elif rr < 15:     score += 0
    elif rr < 21:     score += 1
    elif rr < 30:     score += 2
    else:             score += 3
    temp = row["temp_f"]
    if   temp < 95.0:    score += 2
    elif temp <= 101.1:  score += 0
    else:                score += 2
    return score

def fit_structured_stats(train_df):
    return {
        "vital_medians": {col: float(train_df[col].median()) for col in RAW_VITALS},
        "pain_median":   float(train_df["pain"].median()),
        "age_median":    float(train_df["age"].median()),
    }

def transform_structured(df_in, stats):
    feat = df_in.copy()
    for col in RAW_VITALS:
        feat[col] = feat[col].fillna(stats["vital_medians"][col]).clip(*CLIP_BOUNDS[col])
    feat["pain"] = feat["pain"].fillna(stats["pain_median"]).clip(0.0, 10.0)
    feat["age"]  = feat["age"].fillna(stats["age_median"]).clip(18, 120)
    feat["pain_missing"]      = feat["pain_missing"].astype(int)
    feat["transport_ordinal"] = feat["arrival_transport"].map(TRANSPORT_MAP).fillna(1).astype(int)
    feat["shock_index"]    = (feat["heart_rate"] / feat["sbp"].replace(0, np.nan)).fillna(0.0)
    feat["map"]            = (feat["sbp"] + 2.0 * feat["dbp"]) / 3.0
    feat["pulse_pressure"] = feat["sbp"] - feat["dbp"]
    feat["news2_score"] = feat.apply(compute_news2, axis=1)
    feat["mews_score"]  = feat.apply(compute_mews,  axis=1)
    return feat[STRUCTURED_FEATURES].astype(np.float32)

structured_stats = fit_structured_stats(X_train)
X_train_struct   = transform_structured(X_train, structured_stats)
X_val_struct     = transform_structured(X_val,   structured_stats)
X_test_struct    = transform_structured(X_test,  structured_stats)
print(f"Structured features: {len(STRUCTURED_FEATURES)}")

# ── Class weights ──────────────────────────────────────────────────────────────
train_counts = {int(c): int(n) for c, n in zip(*np.unique(y_train, return_counts=True))}
raw_class_weights = torch.tensor(
    [len(y_train) / (NUM_CLASSES * train_counts[c]) for c in range(NUM_CLASSES)],
    dtype=torch.float32,
)
CLASS_WEIGHTS = torch.sqrt(raw_class_weights)
print(f"Class weights (sqrt-dampened): {[round(x, 4) for x in CLASS_WEIGHTS.tolist()]}")

# ── LightGBM 5-fold cross-fitting ─────────────────────────────────────────────
sqrt_w = np.sqrt(np.array([len(y_train) / (3 * train_counts[c]) for c in range(NUM_CLASSES)]))
sample_weight_map = {c: float(sqrt_w[c]) for c in range(NUM_CLASSES)}

lgbm_params = dict(
    objective="multiclass", num_class=NUM_CLASSES,
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
    reg_alpha=0.5, reg_lambda=2.0, random_state=SEED, n_jobs=-1, verbose=-1,
)

X_train_np = X_train_struct.values.astype(np.float32)
X_val_np   = X_val_struct.values.astype(np.float32)
X_test_np  = X_test_struct.values.astype(np.float32)

oof_probs        = np.zeros((len(X_train_np), NUM_CLASSES), dtype=np.float32)
val_probs_folds  = []
test_probs_folds = []
lgbm_fold_models = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (fit_idx, hold_idx) in enumerate(skf.split(X_train_np, y_train), start=1):
    X_fit,  y_fit  = X_train_np[fit_idx],  y_train[fit_idx]
    X_hold, y_hold = X_train_np[hold_idx], y_train[hold_idx]
    sw_fit = np.array([sample_weight_map[int(l)] for l in y_fit], dtype=np.float32)
    lgbm_model = LGBMClassifier(**lgbm_params)
    lgbm_model.fit(X_fit, y_fit, sample_weight=sw_fit,
                   eval_set=[(X_hold, y_hold)],
                   callbacks=[early_stopping(30, verbose=False)])
    oof_probs[hold_idx] = lgbm_model.predict_proba(X_hold)
    val_probs_folds.append(lgbm_model.predict_proba(X_val_np))
    test_probs_folds.append(lgbm_model.predict_proba(X_test_np))
    lgbm_fold_models.append(lgbm_model)
    hold_f1 = f1_score(y_hold, oof_probs[hold_idx].argmax(axis=1), average="macro")
    print(f"  Fold {fold} | best_iter={lgbm_model.best_iteration_} | holdout macro-F1={hold_f1:.4f}")

lgbm_probs_train = oof_probs
lgbm_probs_val   = np.mean(np.stack(val_probs_folds,  axis=0), axis=0)
lgbm_probs_test  = np.mean(np.stack(test_probs_folds, axis=0), axis=0)
print(f"\nLGBM OOF macro-F1: {f1_score(y_train, lgbm_probs_train.argmax(axis=1), average='macro'):.4f}")
print(f"LGBM Val macro-F1: {f1_score(y_val,   lgbm_probs_val.argmax(axis=1),   average='macro'):.4f}")

## 5. Dataset & DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)


class TriageDataset(Dataset):
    """Tokenized triage_text + LightGBM OOF probabilities + label."""

    def __init__(self, frame, lgbm_probs, labels, tokenizer, max_len):
        self.texts      = frame["triage_text"].tolist()
        self.lgbm_probs = torch.tensor(lgbm_probs, dtype=torch.float32)
        self.labels     = torch.tensor(labels, dtype=torch.long)
        self.tokenizer  = tokenizer
        self.max_len    = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "lgbm_probs":     self.lgbm_probs[idx],
            "label":          self.labels[idx],
        }


train_dataset = TriageDataset(X_train, lgbm_probs_train, y_train, tokenizer, MAX_LEN)
val_dataset   = TriageDataset(X_val,   lgbm_probs_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = TriageDataset(X_test,  lgbm_probs_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

batch = next(iter(train_loader))
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"Effective batch size: {BATCH_SIZE * ACCUMULATION_STEPS}")
print({k: tuple(v.shape) for k, v in batch.items()})

## 7. Model Definition

`LoRAFusionModel`: BioClinicalBERT with LoRA on all 12 layers (768-dim) + LightGBM probs (3-dim)
→ LayerNorm → Linear(771, 96) → GELU → Dropout(0.4) → Linear(96, 3)

LoRA replaces manual layer unfreezing — adapts all layers with ~2M params vs 21M in arch4.

In [ ]:
class LoRAFusionModel(nn.Module):
    def __init__(self, bert_model_name, lora_config, tree_dim=3, num_classes=3,
                 hidden_dim=HEAD_HIDDEN_DIM, dropout=HEAD_DROPOUT):
        super().__init__()
        base_bert  = AutoModel.from_pretrained(bert_model_name)
        self.bert  = get_peft_model(base_bert, lora_config)
        bert_dim   = self.bert.config.hidden_size  # 768
        self.fusion_head = nn.Sequential(
            nn.LayerNorm(bert_dim + tree_dim),
            nn.Linear(bert_dim + tree_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def masked_mean_pool(self, last_hidden_state, attention_mask):
        mask   = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    def forward(self, input_ids, attention_mask, tree_probs):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.masked_mean_pool(out.last_hidden_state, attention_mask)
        fused  = torch.cat([pooled, tree_probs], dim=1)
        return self.fusion_head(fused)


lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)

model = LoRAFusionModel(BERT_MODEL, lora_config).to(DEVICE)
model.bert.gradient_checkpointing_enable()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Trainable %:      {100 * trainable_params / total_params:.1f}%")
model.bert.print_trainable_parameters()

with torch.no_grad():
    logits = model(
        batch["input_ids"].to(DEVICE),
        batch["attention_mask"].to(DEVICE),
        batch["lgbm_probs"].to(DEVICE),
    )
    print(f"Forward pass OK — logits shape: {tuple(logits.shape)}")

## 7. Training Loop

Single-phase training — LoRA adapters + head trained together from epoch 1.  
No layer unfreezing schedule needed.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))

optimizer = AdamW([
    {"params": model.bert.parameters(),         "lr": BERT_LR,  "weight_decay": 0.01},
    {"params": model.fusion_head.parameters(),  "lr": HEAD_LR,  "weight_decay": 0.01},
])

steps_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(0.10 * total_steps))
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")
print(f"BERT LR: {BERT_LR} | Head LR: {HEAD_LR}")

In [ ]:
def compute_critical_f1(labels, preds):
    report = classification_report(labels, preds, labels=[0, 1, 2], output_dict=True, zero_division=0)
    return float(report["0"]["f1-score"])


def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0.0
    preds, labels = [], []
    optimizer.zero_grad()

    for step, batch in enumerate(loader, start=1):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        tree_probs     = batch["lgbm_probs"].to(DEVICE)
        target         = batch["label"].to(DEVICE)

        with torch.amp.autocast(device_type=AUTOCAST_DEVICE, dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            logits = model(input_ids, attention_mask, tree_probs)
            loss   = criterion(logits, target) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if step % ACCUMULATION_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS
        preds.extend(logits.detach().argmax(dim=1).cpu().numpy())
        labels.extend(target.detach().cpu().numpy())

    return {
        "loss":     total_loss / len(loader),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds, labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            tree_probs     = batch["lgbm_probs"].to(DEVICE)
            target         = batch["label"].to(DEVICE)

            with torch.amp.autocast(device_type=AUTOCAST_DEVICE, dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                logits = model(input_ids, attention_mask, tree_probs)
                loss   = criterion(logits, target)

            total_loss += loss.item()
            preds.extend(logits.detach().argmax(dim=1).cpu().numpy())
            labels.extend(target.detach().cpu().numpy())

    return {
        "loss":        total_loss / len(loader),
        "macro_f1":    f1_score(labels, preds, average="macro"),
        "critical_f1": compute_critical_f1(labels, preds),
        "preds":       np.array(preds),
        "labels":      np.array(labels),
    }


def predict_probs(model, loader):
    model.eval()
    outputs = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            tree_probs     = batch["lgbm_probs"].to(DEVICE)
            with torch.amp.autocast(device_type=AUTOCAST_DEVICE, dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                logits = model(input_ids, attention_mask, tree_probs)
            outputs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(outputs)

In [ ]:
best_val_f1     = -1.0
bad_epochs      = 0
history         = []
best_model_path = "/tmp/best_model_arch5.pt"
PATIENCE        = 7   # increased — loss still decreasing at epoch 12, stopping was premature
EPOCHS          = 25  # more room to converge

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_epoch(model, train_loader, optimizer, scheduler, criterion)
    val_metrics   = eval_epoch(model, val_loader, criterion)

    gap      = train_metrics["macro_f1"] - val_metrics["macro_f1"]
    improved = val_metrics["macro_f1"] > best_val_f1

    if improved:
        best_val_f1 = val_metrics["macro_f1"]
        bad_epochs  = 0
        torch.save(model.state_dict(), best_model_path)
        status = "<- best"
    else:
        bad_epochs += 1
        status = f"patience {bad_epochs}/{PATIENCE}"

    history.append({
        "epoch":           epoch,
        "train_loss":      train_metrics["loss"],
        "val_loss":        val_metrics["loss"],
        "train_f1":        train_metrics["macro_f1"],
        "val_f1":          val_metrics["macro_f1"],
        "val_critical_f1": val_metrics["critical_f1"],
        "generalization_gap": gap,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['macro_f1']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['macro_f1']:.4f} | "
        f"critical_f1={val_metrics['critical_f1']:.4f} gap={gap:.4f} | {status}"
    )

    if bad_epochs >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print()
print(f"Best validation macro-F1: {best_val_f1:.4f}")
history_df = pd.DataFrame(history)
history_df

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"],   marker="o", label="val_loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["train_f1"], marker="o", label="train_f1")
axes[1].plot(history_df["epoch"], history_df["val_f1"],   marker="o", label="val_f1")
axes[1].set_title("Macro-F1")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(history_df["epoch"], history_df["generalization_gap"], marker="o",
             color="purple", label="train_f1 - val_f1")
axes[2].plot(history_df["epoch"], history_df["val_critical_f1"],    marker="o",
             color="crimson", label="val L1-Critical F1")
axes[2].set_title("Generalization gap & Critical F1")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle("Architecture 5: BioClinicalBERT + LoRA (Vitals-in-Text)", fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

val_metrics    = eval_epoch(model, val_loader,  criterion)
test_metrics   = eval_epoch(model, test_loader, criterion)
all_test_probs = predict_probs(model, test_loader)

weighted_f1 = f1_score(y_test, test_metrics["preds"], average="weighted")
roc_auc     = roc_auc_score(y_test, all_test_probs, multi_class="ovr", average="macro")

print("Validation macro-F1:   ", round(val_metrics["macro_f1"],  4))
print("Test macro-F1:         ", round(test_metrics["macro_f1"], 4))
print("Validation critical F1:", round(val_metrics["critical_f1"],  4))
print("Test critical F1:      ", round(test_metrics["critical_f1"], 4))
print("Test weighted F1:      ", round(weighted_f1, 4))
print("Test ROC-AUC (OvR):    ", round(roc_auc, 4))
print()

BASELINES = {
    "arch4_v1": {"macro_f1": 0.6661, "critical_f1": 0.5750, "roc_auc": 0.8277},
    "arch5_v3": {"macro_f1": 0.6486, "critical_f1": 0.5517, "roc_auc": 0.8332},
}
print("--- vs baselines ---")
for name, b in BASELINES.items():
    print(f"  {name}:")
    print(f"    macro-F1:    {test_metrics['macro_f1']:.4f} vs {b['macro_f1']:.4f} (delta={test_metrics['macro_f1']-b['macro_f1']:+.4f})")
    print(f"    critical-F1: {test_metrics['critical_f1']:.4f} vs {b['critical_f1']:.4f} (delta={test_metrics['critical_f1']-b['critical_f1']:+.4f})")
    print(f"    ROC-AUC:     {roc_auc:.4f} vs {b['roc_auc']:.4f} (delta={roc_auc-b['roc_auc']:+.4f})")
print()

print("Classification report:")
print(classification_report(y_test, test_metrics["preds"],
                            target_names=CLASS_NAMES, zero_division=0))

summary_df = pd.DataFrame({
    "metric": ["val_macro_f1", "test_macro_f1", "val_critical_f1",
               "test_critical_f1", "test_weighted_f1", "test_roc_auc_ovr_macro"],
    "value":  [val_metrics["macro_f1"], test_metrics["macro_f1"],
               val_metrics["critical_f1"], test_metrics["critical_f1"],
               weighted_f1, roc_auc],
})
summary_df


In [ ]:
cm     = confusion_matrix(y_test, test_metrics["preds"], labels=[0, 1, 2])
cm_pct = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_pct, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=["Critical", "Emergent", "Urgent/LU"],
    yticklabels=["Critical", "Emergent", "Urgent/LU"],
)
plt.title(
    "Confusion Matrix (% of actual class)\n"
    "Arch5: BioClinicalBERT + LoRA (Vitals-in-Text)",
    fontweight="bold"
)
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

print("\nRaw confusion matrix:")
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

## 10. Save Artifacts to S3

In [ ]:
if DEBUG_MODE:
    print("DEBUG MODE — skipping S3 artifact saves.")
else:
    def upload_file(local_path, s3_key):
        s3.upload_file(local_path, S3_BUCKET, s3_key)
        print(f"  Uploaded → s3://{S3_BUCKET}/{s3_key}")

    print(f"Saving artifacts to s3://{S3_BUCKET}/{MODEL_PREFIX}")

    # 1. Model weights (LoRA adapters merged into base for portability)
    merged_model = model.bert.merge_and_unload()
    model_path   = "/tmp/arch5_best_model.pt"
    torch.save({
        "bert_state_dict": merged_model.state_dict(),
        "head_state_dict": model.head.state_dict(),
    }, model_path)
    upload_file(model_path, f"{MODEL_PREFIX}best_model.pt")

    # 2. Training history
    history_path = "/tmp/arch5_history.csv"
    history_df.to_csv(history_path, index=False)
    upload_file(history_path, f"{MODEL_PREFIX}history.csv")

    # 3. Config
    config = {
        "architecture":  "arch5",
        "description":   "BioClinicalBERT + LoRA, vitals serialized in text, no LightGBM",
        "bert_model":    BERT_MODEL,
        "max_len":       MAX_LEN,
        "lora_r":        LORA_R,
        "lora_alpha":    LORA_ALPHA,
        "lora_dropout":  LORA_DROPOUT,
        "lora_targets":  LORA_TARGET_MODULES,
        "n_classes":     NUM_CLASSES,
        "text_format":   "CC_2x + Vitals + Pain + Transport + HPI",
        "changes_vs_arch4_v1": [
            "Vitals serialized into text (HR, BP, RR, SpO2, Temp, Pain, Transport)",
            "LoRA fine-tuning of all 12 layers (~2M trainable params vs 21M in arch4)",
            "LightGBM removed — no fusion branch",
            "Single-phase training — no head warmup / layer unfreeze schedule",
        ],
        "test_metrics": {
            "macro_f1":    round(test_metrics["macro_f1"],  4),
            "critical_f1": round(test_metrics["critical_f1"], 4),
            "weighted_f1": round(weighted_f1, 4),
            "roc_auc_ovr": round(roc_auc, 4),
        },
        "best_val_f1": round(best_val_f1, 4),
    }
    config_path = "/tmp/arch5_config.json"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    upload_file(config_path, f"{MODEL_PREFIX}config.json")

    print()
    print("All artifacts saved.")
    print(f"Config: {json.dumps(config, indent=2)}")